<a href="https://colab.research.google.com/github/abregation/vik_data/blob/main/Corpus_Crack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install spacy and the medium-sized Spanish model
!pip install spacy pandas openpyxl
!python -m spacy download es_core_news_md

import spacy
import pandas as pd
import io
from google.colab import files

# Load the Spanish model
nlp = spacy.load("es_core_news_md")
print("Libraries and Spanish model loaded successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 18.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Libraries and Spanish model loaded successfully.


In [ ]:
print("Please upload your CSV or Excel file:")
uploaded = files.upload()

# Identify the file name and extension
file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
else:
    df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print(f"Successfully loaded: {file_name}")
df.head()

Please upload your CSV or Excel file:


Saving Kwic_Mimama_CdlV_Depurado.xlsx to Kwic_Mimama_CdlV_Depurado (2).xlsx
Successfully loaded: Kwic_Mimama_CdlV_Depurado (2).xlsx


,corpus
0,"Cuando estaban muchachos, ellos, mi mamá y mi ..."
1,éramos amigos de Etnio. Me fui para Bogotá esa...
2,yo era muy delicada. Me pagó una pieza. Entonc...
3,"lo amo y bendigo a mi viejo, pero cuando yo er..."
4,"en cambio en esas zonas no, en esas zonas tien..."


In [ ]:
target_column = 'corpus'

if target_column not in df.columns:
    print(f"Column '{target_column}' not found.")
    print("Available columns:", df.columns.tolist())
    # Manual override: change 'text_column_name' to your actual column name
    # target_column = 'text_column_name'
else:
    print(f"Target column '{target_column}' identified.")

Target column 'corpus' identified.


In [ ]:
def process_text(text):
    if pd.isna(text):
        return "", "", "", "", "", ""

    doc = nlp(str(text))

    # Parts of Speech (POS) extraction
    sustantivos = [token.text for token in doc if token.pos_ == "NOUN"]
    adjetivos = [token.text for token in doc if token.pos_ == "ADJ"]
    verbos = [token.text for token in doc if token.pos_ == "VERB"]
    adverbios = [token.text for token in doc if token.pos_ == "ADV"]

    # Named Entity Recognition (NER) extraction
    nombres = [ent.text for ent in doc.ents if ent.label_ == "PER"]
    lugares = [ent.text for ent in doc.ents if ent.label_ == "LOC"]

    return (
        ", ".join(sustantivos),
        ", ".join(adjetivos),
        ", ".join(verbos),
        ", ".join(adverbios),
        ", ".join(nombres),
        ", ".join(lugares)
    )

print("Processing text... (this may take a moment depending on the size)")

# Apply the function to the dataframe
df[['sustantivos', 'adjetivos', 'verbos', 'adverbios', 'nombres_personas', 'lugares']] = df[target_column].apply(
    lambda x: pd.Series(process_text(x))
)

print("Analysis complete.")
df.head()

Processing text... (this may take a moment depending on the size)
Analysis complete.


,corpus,sustantivos,adjetivos,verbos,adverbios,nombres_personas,lugares
0,"Cuando estaban muchachos, ellos, mi mamá y mi ...","muchachos, mamá","familiar, juntos",criaron,prácticamente,,
1,éramos amigos de Etnio. Me fui para Bogotá esa...,"amigos, mamá, familia, enero",,"fui, desaparecen, Quería, pasar, Llegué","No, así","Etnio, Etnio, Quería","Llegué, Cali"
2,yo era muy delicada. Me pagó una pieza. Entonc...,"pieza, días, mamá, papá, barriga, casa","delicada, encerrada, preñada","pagó, vivía, llorando, pensaba, llegar","muy, Entonces, allá, No",,
3,"lo amo y bendigo a mi viejo, pero cuando yo er...","chico, completo, mamá, abuela, momento","bendigo, viejo, vulnerables","amo, desapareció",Entonces,,
4,"en cambio en esas zonas no, en esas zonas tien...","cambio, zonas, zonas, acceso, mamá, abuela, he...",,"tienen, saque, moverlas","no, más, Entonces, apenas, allá",,


In [ ]:
def count_items_in_string(text):
    if pd.isna(text) or text == "":
        return 0
    return len(text.split(', '))

# Apply the counting function to create new columns
df['num_sustantivos'] = df['sustantivos'].apply(count_items_in_string)
df['num_adjetivos'] = df['adjetivos'].apply(count_items_in_string)
df['num_verbos'] = df['verbos'].apply(count_items_in_string)
df['num_adverbios'] = df['adverbios'].apply(count_items_in_string)

print("New count columns added.")
df.head()

New count columns added.


,corpus,sustantivos,adjetivos,verbos,adverbios,nombres_personas,lugares,num_sustantivos,num_adjetivos,num_verbos,num_adverbios
0,"Cuando estaban muchachos, ellos, mi mamá y mi ...","muchachos, mamá","familiar, juntos",criaron,prácticamente,,,2,2,1,1
1,éramos amigos de Etnio. Me fui para Bogotá esa...,"amigos, mamá, familia, enero",,"fui, desaparecen, Quería, pasar, Llegué","No, así","Etnio, Etnio, Quería","Llegué, Cali",4,0,5,2
2,yo era muy delicada. Me pagó una pieza. Entonc...,"pieza, días, mamá, papá, barriga, casa","delicada, encerrada, preñada","pagó, vivía, llorando, pensaba, llegar","muy, Entonces, allá, No",,,6,3,5,4
3,"lo amo y bendigo a mi viejo, pero cuando yo er...","chico, completo, mamá, abuela, momento","bendigo, viejo, vulnerables","amo, desapareció",Entonces,,,5,3,2,1
4,"en cambio en esas zonas no, en esas zonas tien...","cambio, zonas, zonas, acceso, mamá, abuela, he...",,"tienen, saque, moverlas","no, más, Entonces, apenas, allá",,,8,0,3,5


In [ ]:
output_file = "linguistic_analysis_results.csv"

# Save to CSV
df.to_csv(output_file, index=False, encoding='utf-8-sig')

# Download the file
files.download(output_file)

print(f"File '{output_file}' is ready and downloading.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'linguistic_analysis_results.csv' is ready and downloading.
